# Electricity Agreement Optimizer
Compare provider agreements against 12 months of electricity usage. Each company gets a specialist agent; the supervisor checks all results before a plan enters the ranking.

**Start in demo mode.** Demo contracts and negotiation questions are fictional fixtures, not GPT-5 outputs. Live mode calls GPT-5 for extraction and independent review.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "electricity_ao").exists():
    ROOT = ROOT / "Electricity_AO_Update"
if not (ROOT / "electricity_ao").exists():
    raise RuntimeError("Open Jupyter from Electricity_AO_Update or its parent folder")
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from electricity_ao.agents import OpenAIBackend
from electricity_ao.demo import create_demo
from electricity_ao.documents import load_usage
from electricity_ao.models import Preferences, ProviderConfig
from electricity_ao.workflow import build_graph

## 1. Select inputs
Leave `MODE = "demo"` for an API-free run. For real comparisons, follow README setup, configure PDFs in `data/private/providers.json`, supply usage, then choose `"live"`. Each provider is a separate graph node. Switching cost is a one-time current-contract exit/setup cost applied to each candidate.

In [2]:
MODE = "live"
PROVIDERS_PATH = ROOT / "data/comparable_samples/providers.json"
USAGE_PATH = ROOT / "data/usage_example.csv"
preferences = Preferences(
    max_term_months=24,
    minimum_renewable_percent=0,
    switching_cost_usd=0,
)

if MODE == "demo":
    providers, backend = create_demo(ROOT)
    usage = load_usage(ROOT / "data/usage_example.csv")
elif MODE == "live":
    providers = [ProviderConfig.model_validate(row)
                 for row in json.loads(PROVIDERS_PATH.read_text(encoding="utf-8"))]
    for provider in providers:
        provider.contract_path = str(ROOT / provider.contract_path)
    usage = load_usage(USAGE_PATH)
    backend = OpenAIBackend()
else:
    raise ValueError("MODE must be demo or live")

display(pd.DataFrame([month.model_dump() for month in usage.months]))
display(pd.DataFrame([provider.model_dump() for provider in providers]))

,month,kwh
0,2025-09-01,1400.0
1,2025-10-01,1000.0
2,2025-11-01,750.0
3,2025-12-01,850.0
4,2026-01-01,950.0
5,2026-02-01,800.0
6,2026-03-01,700.0
7,2026-04-01,850.0
8,2026-05-01,1100.0
9,2026-06-01,1500.0


,provider_id,company,contract_path
0,gexa_saver_12_centerpoint,"Gexa Energy, LP",c:\Users\allen\VSCode_Projects\Electricity_AO_...
1,gexa_saver_24_centerpoint,"Gexa Energy, LP",c:\Users\allen\VSCode_Projects\Electricity_AO_...


## 2. Build the supervisor graph
Specialists execute in parallel. The final supervisor waits for every provider, checks evidence and deterministic estimates, then independently reviews the source documents in live mode.

In [3]:
graph = build_graph(providers, backend, mode=MODE)
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor_dispatch(supervisor_dispatch)
	provider_gexa_saver_12_centerpoint(provider_gexa_saver_12_centerpoint)
	provider_gexa_saver_24_centerpoint(provider_gexa_saver_24_centerpoint)
	supervisor_review(supervisor_review)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor_dispatch;
	provider_gexa_saver_12_centerpoint --> supervisor_review;
	provider_gexa_saver_24_centerpoint --> supervisor_review;
	supervisor_dispatch --> provider_gexa_saver_12_centerpoint;
	supervisor_dispatch --> provider_gexa_saver_24_centerpoint;
	supervisor_review --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 3. Run extraction and review
Live mode sends contract text to OpenAI and makes billable calls. Failed or unsupported plans remain visible in the review table and are excluded from ranking.

In [4]:
state = graph.invoke({"usage": usage, "preferences": preferences, "results": []})
report = state["report"]
display(pd.DataFrame([review.model_dump() for review in report.reviews]))
print("Recommended provider:", report.recommended_provider_id or "None — resolve review issues")

,provider_id,approved,issues
0,gexa_saver_12_centerpoint,False,[Unverifiable source quote for early_terminati...
1,gexa_saver_24_centerpoint,False,[Unverifiable source quote for early_terminati...


Recommended provider: None — resolve review issues


## 4. Inspect the extracted terms and evidence
Validate rates, delivery fees, credits, renewal language, and page quotations before relying on live output.

In [5]:
for result in sorted(state["results"], key=lambda result: result.provider_id):
    print("\nPROVIDER:", result.provider_id)
    if result.terms:
        display(pd.DataFrame([result.terms.model_dump(exclude={"evidence"})]))
        display(pd.DataFrame([item.model_dump() for item in result.terms.evidence]))
    else:
        print(result.issues)


PROVIDER: gexa_saver_12_centerpoint


,provider,plan_name,pricing_type,currency,term_months,energy_cents_per_kwh,delivery_cents_per_kwh,monthly_base_usd,monthly_delivery_usd,credit_usd,credit_min_kwh,credit_max_kwh,early_termination_usd,renewable_percent,renewal_terms,unresolved_terms
0,"Gexa Energy, LP",Gexa Energy Saver 12,fixed,USD,12,12.95,4.2392,0.0,4.9,None,None,None,150.0,100.0,None,[Paperless-only eligibility: customer must mai...


,field,page,quote
0,plan_name,1,Gexa Energy Saver 12
1,pricing_type,1,Type of Product: ...
2,term_months,1,Contract Term: 1...
3,energy_cents_per_kwh,1,Energy Charge ...
4,monthly_delivery_usd,1,TDU Delivery Charges ...
5,delivery_cents_per_kwh,1,TDU Delivery Charges ...
6,monthly_base_usd,1,The price you pay each month includes the Ener...
7,early_termination_usd,1,"Yes, Gexa will assess a $150.00 early terminat..."
8,renewable_percent,2,This product is 100% renewable
9,unresolved_terms,1,This is a paperless product. All bills and con...



PROVIDER: gexa_saver_24_centerpoint


,provider,plan_name,pricing_type,currency,term_months,energy_cents_per_kwh,delivery_cents_per_kwh,monthly_base_usd,monthly_delivery_usd,credit_usd,credit_min_kwh,credit_max_kwh,early_termination_usd,renewable_percent,renewal_terms,unresolved_terms
0,"Gexa Energy, LP",Gexa Energy Saver 24,fixed,USD,24,12.45,4.2392,None,4.9,None,None,None,295.0,100.0,None,[EFL does not provide renewal terms; see Terms...


,field,page,quote
0,pricing_type,1,Type of Product: ...
1,term_months,1,Contract Term: 2...
2,energy_cents_per_kwh,1,Energy Charge ...
3,monthly_delivery_usd,1,TDU Delivery Charges ...
4,delivery_cents_per_kwh,1,TDU Delivery Charges ...
5,early_termination_usd,1,"Yes, Gexa will assess a $295.00 early terminat..."
6,renewable_percent,2,This product is 100% renewable


## 5. Compare approved plans
Amounts are pre-tax estimates using the same historical usage. This is a lowest-cost comparison subject to your preferences, not a guarantee of future bills.

In [6]:
ranking = pd.DataFrame([{
    "provider": item.provider_id,
    "plan": item.plan_name,
    "annual_usd": item.annual_usd,
    "first_year_usd": item.first_year_usd,
    "effective_cents_per_kwh": item.effective_cents_per_kwh,
} for item in report.ranking], columns=[
    "provider", "plan", "annual_usd", "first_year_usd", "effective_cents_per_kwh"
])
if report.ranking:
    display(ranking)
    monthly = pd.DataFrame({
        item.provider_id: [month.cost_usd for month in item.monthly]
        for item in report.ranking
    }, index=[month.month.strftime("%Y-%m") for month in usage.months])
    ax = monthly.plot(marker="o", figsize=(11, 4), ylabel="Estimated monthly cost (USD, pre-tax)",
                      xlabel="Historical usage month", title="Your usage under each approved plan")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("### No plans qualified for the cost ranking"))
    print("The workflow completed, but every plan was withheld during validation or supervisor review.")
    print("Review the issues below, then supply complete EFL and Terms of Service documents where needed.")
    display(pd.DataFrame([
        {"provider": review.provider_id, "issues": " | ".join(review.issues)}
        for review in report.reviews
    ]))

### No plans qualified for the cost ranking

The workflow completed, but every plan was withheld during validation or supervisor review.
Review the issues below, then supply complete EFL and Terms of Service documents where needed.


,provider,issues
0,gexa_saver_12_centerpoint,Unverifiable source quote for early_terminatio...
1,gexa_saver_24_centerpoint,Unverifiable source quote for early_terminatio...


## 6. Negotiation questions and limitations
These are suggested questions to discuss with a provider; no agreement is changed or message sent.

In [7]:
for question in report.negotiation_questions:
    print("•", question)
for limitation in report.limitations:
    print("Note:", limitation)

• Please confirm there is no Gexa base monthly charge on Gexa Energy Saver 12 (CenterPoint) beyond the per-kWh Energy Charge and CenterPoint TDU charges of $4.90/month and 4.2392¢/kWh.
• Please confirm there is no Gexa base monthly charge on Gexa Energy Saver 24 (CenterPoint) beyond the per-kWh Energy Charge and CenterPoint TDU charges of $4.90/month and 4.2392¢/kWh.
• Provide the current “Summary of Gexa Energy Non-Recurring Charges” from the Terms of Service (amounts and conditions for connection, disconnection, paper bill, payment processing/NSF, late fees, etc.).
• What specific fee(s) and dollar amount(s) apply if a valid email is not maintained on these paperless products? Are any of those fees recurring?
• Is the service address subject to the CenterPoint Underground Facilities and Cost Recovery municipal charge mentioned in the EFL? If yes, what is the exact amount per billing cycle?
• Do either plan include any minimum-usage fees or bill credits at specific kWh thresholds? The

## 7. Export your comparison
Outputs may contain contract terms. The outputs folder is excluded from Git.

In [8]:
output_dir = ROOT / "outputs"
output_dir.mkdir(exist_ok=True)
(output_dir / "comparison.json").write_text(report.model_dump_json(indent=2), encoding="utf-8")
ranking.to_csv(output_dir / "ranking.csv", index=False)
(output_dir / "extractions.json").write_text(
    json.dumps([result.model_dump(mode="json") for result in state["results"]], indent=2),
    encoding="utf-8",
)
print("Saved comparison.json, ranking.csv, and extractions.json to", output_dir)

Saved comparison.json, ranking.csv, and extractions.json to c:\Users\allen\VSCode_Projects\Electricity_AO_Update\outputs
